# Image Classification Pipeline using PCA (Eigenfaces) & SVM
### Machine Learning Assignment #4
This notebook implements an end-to-end, high-performance image classification pipeline using **Principal Component Analysis (PCA)** for dimensionality reduction (conceptually known as "Eigenfaces") and **Support Vector Machines (SVM)** with a Radial Basis Function (RBF) kernel for classification.

---

## Part 1: Theoretical Analysis (PCA & Feature Scaling)

### 1.1 What is PCA?
**Principal Component Analysis (PCA)** is an unsupervised linear dimensionality reduction technique. It projects high-dimensional data onto a lower-dimensional subspace while maximizing the variance of the projected data (or equivalently, minimizing the reconstruction error).

#### Mathematical Foundations:
Given a centered data matrix $X \in \mathbb{R}^{N \times D}$ (where $N$ is the number of samples and $D$ is the original feature dimension):
1. **Covariance Matrix**: We compute the sample covariance matrix $\Sigma \in \mathbb{R}^{D \times D}$ as:
   $$\Sigma = \frac{1}{N-1} X^T X$$
2. **Eigendecomposition**: We find the eigenvalues $\lambda_i$ and corresponding orthonormal eigenvectors $v_i$ such that:
   $$\Sigma v_i = \lambda_i v_i$$
   The eigenvectors $v_i$ represent the **Principal Components (PCs)**, which define the directions of maximum variance in the data. The eigenvalues $\lambda_i$ represent the amount of variance explained by each PC.
3. **Projection**: To project the data into a lower-dimensional space of dimension $K \ll D$, we construct a projection matrix $W_K = [v_1, v_2, \dots, v_K] \in \mathbb{R}^{D \times K}$ from the eigenvectors corresponding to the $K$ largest eigenvalues. The reduced dataset is:
   $$Z = X W_K \in \mathbb{R}^{N \times K}$$

---

### 1.2 Why is Feature Scaling Important Before PCA?
PCA is highly sensitive to the relative scale of the features because it is based on **maximizing variance**. 

1. **Scale Dependency**: Variance is measured in squared units of the feature's scale. If one feature has a scale that is orders of magnitude larger than another (e.g., house prices in thousands vs. number of bedrooms in single digits), its calculated variance will dominate the covariance matrix $\Sigma$.
2. **Directional Bias**: The eigenvectors will align almost entirely along the directions of the unscaled features with the largest numerical ranges, regardless of whether those features contain meaningful information. PCA will treat these directions as the principal components, discarding the variation in smaller-scale features.
3. **Solution**: Applying **Standardization** (scaling features to have a mean of 0 and a standard deviation of 1) ensures that every feature contributes equally to the analysis, allowing PCA to find the true structural directions of variance rather than scale-biased ones.

#### Numerical Example:
Imagine two features:
- $x_1$: Height in meters (range: 1.5m to 2.0m, variance $\approx 0.02$)
- $x_2$: Weight in grams (range: 50,000g to 100,000g, variance $\approx 2.5 \times 10^8$)

If we run PCA without scaling, the covariance along $x_2$ is millions of times larger than along $x_1$. The first principal component will lie almost perfectly parallel to $x_2$, and the variation in $x_1$ (height) will be completely ignored as noise. Standardizing both to $\mu=0, \sigma=1$ puts them on equal footing.

---

### 1.3 Advantages and Disadvantages of PCA

| Advantages | Disadvantages |
| :--- | :--- |
| **1. Dimensionality & Noise Reduction**: Eliminates highly correlated or redundant features (collinearity), and discards components with low eigenvalues which typically represent random noise. | **1. Loss of Interpretability**: The new principal components are linear combinations of the original features (e.g., $0.34 \times \text{pixel\_1} - 0.12 \times \text{pixel\_2} \dots$). They no longer have individual physical meaning. |
| **2. Computational & Storage Efficiency**: Reducing features from thousands to a few dozen dramatically speeds up downstream model training (like SVM) and reduces memory footprints. | **2. Linear Assumption**: PCA assumes that the underlying data structure is linear. If the true data manifold is non-linear (e.g., a Swiss Roll), PCA will fail to capture the correct representation (requiring Kernel PCA instead). |
| **3. Avoids Overfitting**: By reducing the ratio of features to samples ($D/N$), PCA helps mitigate the "Curse of Dimensionality," leading to better generalized models. | **3. Information Loss**: By dropping components with smaller eigenvalues, some discriminative details might be discarded, especially if the target classes are separated along low-variance directions. |

---

### 1.4 What Type of Dataset is Required for PCA?
PCA performs best when the dataset satisfies the following conditions:
1. **Continuous Numeric Features**: Features must be quantitative (interval or ratio scale) because PCA relies on computing means, standard deviations, and covariance matrices. It is not directly applicable to categorical data (e.g., gender, country) without extensions like Multiple Correspondence Analysis (MCA).
2. **Linear Relationships & Multicollinearity**: The features should exhibit correlation. If features are completely uncorrelated, the covariance matrix will be diagonal, each principal component will just be a scaled original feature, and no dimensionality reduction will be achieved.
3. **High-Dimensional Data**: It is most useful when the number of features $D$ is large, or when $D > N$, where models are highly prone to overfitting.
4. **No Severe Outliers**: Outliers can distort the sample mean and variance significantly, causing PCA to align principal components toward the outliers to maximize variance, ruining the projection for the rest of the data.

## Part 2: Implementation of the Image Classification Pipeline

We will now run the complete pipeline using the **Labeled Faces in the Wild (LFW)** dataset. Let's start by importing our libraries.

In [ ]:
import os
import ssl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Bypass SSL verification issues on macOS for scikit-learn download
try:
    ssl._create_default_https_context = ssl._create_unverified_context
except AttributeError:
    pass

# Set styled plotting
sns.set_theme(style="white", palette="muted")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "figure.titlesize": 16
})
print("All libraries successfully imported!")

### Step 1: Loading & Exploring the LFW Dataset

In [ ]:
print("Loading dataset...")
# Load faces of individuals with at least 70 images to ensure a balanced classification task
lfw_people = fetch_lfw_people(min_faces_per_person=70, resize=0.4, download_if_missing=True)

n_samples, h, w = lfw_people.images.shape
X = lfw_people.data
y = lfw_people.target
target_names = lfw_people.target_names
n_classes = target_names.shape[0]

print(f"Total samples: {n_samples}")
print(f"Image size: {h}x{w} pixels = {X.shape[1]} features")
print(f"Classes: {n_classes} target individuals")

#### Class Distribution Analysis

In [ ]:
plt.figure(figsize=(10, 5))
counts = [np.sum(y == i) for i in range(n_classes)]
sns.barplot(x=counts, y=target_names, palette="viridis")
plt.title("Class Distribution (Number of Face Images per Person)", weight="bold")
plt.xlabel("Count")
plt.ylabel("Person")
plt.tight_layout()
plt.show()

#### Visualize Sample Face Images

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(12, 8))
fig.suptitle("Sample Face Images from LFW Dataset", weight="bold")
for i, ax in enumerate(axes.flat):
    ax.imshow(lfw_people.images[i], cmap="gray")
    ax.set_title(target_names[y[i]], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Step 2: Stratified Split & Feature Scaling
Since our classes are imbalanced (George W. Bush has 530 images while Hugo Chavez has 71), using a **Stratified Train/Test Split** is crucial to maintain identical class ratios in both splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Scale pixel features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled Train shape: {X_train_scaled.shape}")
print(f"Scaled Test shape: {X_test_scaled.shape}")

### Step 3: Dimensionality Reduction using PCA (Eigenfaces)
Applying PCA yields the "Eigenfaces" which are standard face bases. We use `whiten=True` which scales each component output to unit variance, a standard trick to improve SVM performance.

In [ ]:
n_components = 150
print(f"Fitting PCA with {n_components} components...")
pca = PCA(n_components=n_components, svd_solver="randomized", whiten=True, random_state=42).fit(X_train_scaled)

X_train_pca = pca.transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

explained_variance = np.cumsum(pca.explained_variance_ratio_)
print(f"Total explained variance by top {n_components} components: {explained_variance[-1]*100:.2f}%")

#### Plot Explained Variance Curve

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, n_components + 1), explained_variance, marker='o', linestyle='-', color='#4A90E2', markevery=10)
plt.axhline(y=0.90, color='r', linestyle='--', alpha=0.7, label='90% Explained Variance')
plt.title("Cumulative Explained Variance vs. Number of Principal Components", weight="bold")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

#### Visualize Eigenfaces (First 15 Principal Components)
These represent the structural variations across human faces found in LFW (such as lighting angle, eye shape, nose structure, hair).

In [ ]:
eigenfaces = pca.components_.reshape((n_components, h, w))
fig, axes = plt.subplots(3, 5, figsize=(12, 8))
fig.suptitle("Eigenfaces: Visualizing the Principal Components", weight="bold")
for i, ax in enumerate(axes.flat):
    ax.imshow(eigenfaces[i], cmap="plasma")
    ax.set_title(f"PC {i+1}", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Step 4: Hyperparameter Tuning of RBF SVM Classifier
We use cross-validated `GridSearchCV` to optimize the soft margin parameter $C$ and kernel bandwidth $\gamma$.

In [ ]:
param_grid = {
    "C": [1e2, 5e2, 1e3, 5e3, 1e4, 5e4],
    "gamma": [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05]
}

svc = SVC(kernel="rbf", class_weight="balanced", random_state=42)
grid_search = GridSearchCV(svc, param_grid, cv=5, n_jobs=-1, verbose=1)
grid_search.fit(X_train_pca, y_train)

best_svm = grid_search.best_estimator_
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best Cross-Validation score: {grid_search.best_score_*100:.2f}%")

### Step 5: Test Set Model Evaluation

In [ ]:
y_pred = best_svm.predict(X_test_pca)
accuracy = accuracy_score(y_test, y_pred)

print(f"Overall Test Set Accuracy: {accuracy*100:.2f}%\n")
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

#### Heatmapped Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=target_names, yticklabels=target_names,
    cbar=True, square=True
)
plt.title("Confusion Matrix Heatmap (SVM Classifier)", weight="bold", pad=20)
plt.xlabel("Predicted Label", labelpad=10)
plt.ylabel("True Label", labelpad=10)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### Step 6: Prediction Diagnostics Gallery

In [ ]:
fig, axes = plt.subplots(4, 6, figsize=(14, 10))
fig.suptitle("SVM Face Recognition Diagnostics (Green = Correct, Red = Incorrect)", weight="bold", fontsize=15)

for i, ax in enumerate(axes.flat):
    # X_test[i] is the unscaled original pixel vector, which we reshape back to its 2D image shape
    ax.imshow(X_test[i].reshape(h, w), cmap="gray")
    
    pred_name = target_names[y_pred[i]].split()[-1]
    true_name = target_names[y_test[i]].split()[-1]
    
    # Green if prediction matches actual label, Red otherwise
    color = "green" if y_pred[i] == y_test[i] else "red"
    
    ax.set_title(f"Pred: {pred_name}\nTrue: {true_name}", color=color, fontsize=9, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.show()